In [5]:
!pip install google-generativeai

In [ ]:
from google import genai
from google.genai import types

In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY_1")

client = genai.Client(api_key=api_key)
model = client.models

In [40]:
import requests
from bs4 import BeautifulSoup

def extract_content_from_url(url, tag_name=None, class_name=None, id_name=None):
    """
    Trích xuất nội dung văn bản từ một URL, tùy chọn lọc theo thẻ, lớp hoặc ID HTML.

    Args:
        url (str): Đường dẫn URL của trang web.
        tag_name (str, optional): Tên thẻ HTML muốn tìm (ví dụ: 'p', 'div', 'h1').
        class_name (str, optional): Tên lớp CSS của thẻ muốn tìm.
        id_name (str, optional): ID của thẻ muốn tìm.

    Returns:
        list: Danh sách các chuỗi văn bản được trích xuất.
              Nếu không có tiêu chí lọc, nó sẽ trả về tất cả văn bản đọc được.
    """
    try:
        # Tải nội dung trang web
        response = requests.get(url)
        response.raise_for_status()  # Ném lỗi cho mã trạng thái HTTP không thành công

        # Phân tích cú pháp HTML
        soup = BeautifulSoup(response.text, 'html.parser')

        extracted_texts = []

        if tag_name:
            # Tìm tất cả các thẻ khớp với tiêu chí
            elements = soup.find_all(tag_name, class_=class_name, id=id_name)
            for element in elements:
                # Lấy văn bản từ mỗi phần tử, loại bỏ các khoảng trắng thừa
                text = element.get_text(separator=' ', strip=True)
                if text:
                    extracted_texts.append(text)
        else:
            # Nếu không có tiêu chí lọc, lấy tất cả văn bản có thể đọc được
            # Loại bỏ các thẻ script và style để tránh mã nguồn
            for script_or_style in soup(["script", "style"]):
                script_or_style.extract() # loại bỏ chúng

            text = soup.get_text(separator=' ', strip=True)
            extracted_texts.append(text)

        return extracted_texts

    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi truy cập URL: {e}")
        return []
    except Exception as e:
        print(f"Có lỗi xảy ra: {e}")
        return []

# --- Cách sử dụng ---
url = 'https://thanhtra.com.vn/phong-chong-tham-nhung/ho-so-tu-lieu/thong-tin-6-dai-an-tham-nhung-dien-hinh-va-so-luong-can-bo-bi-khoi-to-ky-luat-217953.html'
full_text = extract_content_from_url(url)

# Lấy thông tin chính của bài báo ra (ngoại trừ liên hệ, ...)
prompt = f"""Dưới đây là nội dung một trang báo. Hãy giữ nguyên văn bản bài báo chính, bao gồm tiêu đề, nội dung bài viết và các đoạn thông tin thuộc phần nội dung báo chí gốc, nhưng loại bỏ tất cả các phần không liên quan, cụ thể là:

Thông tin liên hệ tòa soạn, địa chỉ, số điện thoại, email

Tên các chuyên mục, từ khóa, chủ đề

Quảng cáo, fanpage, bản quyền

Các bài viết liên quan, tin đọc nhiều, tin khác, các đường dẫn khác

Phần chân trang, thông tin pháp lý, hiệp hội chủ quản

Văn bản đầu ra phải là nguyên văn nội dung bài báo, không rút gọn, không tóm tắt, không thêm bớt. Chỉ xóa những phần không liên quan như mô tả ở trên.

{full_text}
"""

response = model.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)

print(response.text)


Thông tin 6 đại án tham nhũng điển hình và số lượng cán bộ bị khởi tố, kỷ luật
Hương Giang Thứ tư, 22/11/2023 - 20:58
(Thanh tra) - Phó Trưởng Ban Nội chính Trung ương Đặng Văn Dũng thông tin cụ thể 6 đại án tham nhũng điển hình sai phạm trong các lĩnh vực như AIC, FLC, Vạn Thịnh Phát, Ngân hàng SCB… cũng như số lượng cán bộ bị khởi tố, kỷ luật.

Phó Trưởng Ban Nội chính Trung ương Đặng Văn Dũng. Ảnh: Đ.X

Sau khi Tổng Bí thư Nguyễn Phú Trọng chủ trì cuộc họp của Thường trực Ban Chỉ đạo Trung ương về phòng, chống tham nhũng, tiêu cực, Ban Nội chính Trung ương đã thông tin về kết quả cuộc họp.

Lần đầu tiên khởi tố tội tham ô tài sản với chủ doanh nghiệp ngoài Nhà nước

Theo Phó Ban Nội chính Trung ương Đặng Văn Dũng, tại cuộc họp, Thường trực Ban Chỉ đạo Trung ương phòng, chống tham nhũng, tiêu cực đánh giá tiến độ điều tra, xử lý các vụ án, vụ việc cơ bản đáp ứng yêu cầu của Ban Chỉ đạo, có vụ vượt kế hoạch đề ra. Nhiều vụ án, vụ việc được khởi tố mới, mở rộng điều tra, kiên quyết xử 